In [ ]:
%pip -q install groq streamlit plotly pandas==2.2.3 pyngrok

In [ ]:
import os
import json
import pandas as pd
import plotly.graph_objects as go

from getpass import getpass
from groq import Groq

print("✅ Libraries imported successfully")

In [ ]:
GROQ_API_KEY = getpass("Enter your Groq API key: ")

In [ ]:
client = Groq(
    api_key=GROQ_API_KEY
)

print("✅ Groq client created successfully")

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain prompt engineering in one sentence."
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain prompt engineering."
        }
    ]
)

In [ ]:
response.choices[0].message.content

In [ ]:
SCENARIOS = {

    "Business": {

        "Beginner": {
            "title": "Business Email",
            "description": """
Create a professional email explaining a one-week
project delay to a client.
"""
        },

        "Intermediate": {
            "title": "Meeting Summary",
            "description": """
Convert meeting notes into a structured summary
containing decisions, action items, owners and deadlines.
"""
        },

        "Advanced": {
            "title": "Executive Business Report",
            "description": """
Analyze quarterly business performance and create
an executive report containing insights, risks,
trends and recommendations.
"""
        }
    },

    "HR": {

        "Beginner": {
            "title": "Job Description",
            "description": """
Create a job description for a Python Developer.
"""
        },

        "Intermediate": {
            "title": "Interview Questions",
            "description": """
Generate structured interview questions for an AI/ML Engineer.
"""
        },

        "Advanced": {
            "title": "Candidate Evaluation",
            "description": """
Evaluate candidate interview responses against predefined
competencies and provide structured feedback.
"""
        }
    },

    "Marketing": {

        "Beginner": {
            "title": "Social Media Post",
            "description": """
Create a LinkedIn post announcing a new technology product.
"""
        },

        "Intermediate": {
            "title": "Marketing Campaign",
            "description": """
Design a marketing campaign for a new technology product.
"""
        },

        "Advanced": {
            "title": "Go-To-Market Strategy",
            "description": """
Analyze a target market and create a complete
go-to-market strategy.
"""
        }
    },

    "Data Analytics": {

        "Beginner": {
            "title": "Sales Summary",
            "description": """
Summarize monthly sales data.
"""
        },

        "Intermediate": {
            "title": "Business Insights",
            "description": """
Analyze sales data and identify trends,
patterns and anomalies.
"""
        },

        "Advanced": {
            "title": "Executive Analytics",
            "description": """
Analyze a business dataset and produce actionable
executive insights.
"""
        }
    },

    "Software Development": {

        "Beginner": {
            "title": "Python Function",
            "description": """
Generate a Python function to calculate factorial.
"""
        },

        "Intermediate": {
            "title": "REST API",
            "description": """
Create a REST API using Python and FastAPI.
"""
        },

        "Advanced": {
            "title": "Code Review",
            "description": """
Perform a detailed code review covering security,
performance, maintainability and coding standards.
"""
        }
    }
}

print("Domains:")
for domain in SCENARIOS:
    print("•", domain)

In [ ]:
CRITERIA = {

    "Role":
        "Does the prompt clearly define who the AI should act as?",

    "Goal":
        "Does the prompt clearly state what the AI needs to accomplish?",

    "Context":
        "Does the prompt provide sufficient background information?",

    "Constraints":
        "Does the prompt define rules, limitations or boundaries?",

    "Format":
        "Does the prompt specify the expected output structure or format?",

    "Style":
        "Does the prompt specify the desired tone, style or communication approach?"
}

for criterion, description in CRITERIA.items():

    print(f"{criterion}:")
    print(f"  {description}")
    print()

In [ ]:
def simple_evaluate(prompt):

    response = client.chat.completions.create(

        model="openai/gpt-oss-20b",

        messages=[

            {
                "role": "system",
                "content": """
You are an expert prompt engineering evaluator.

Evaluate the user's prompt based on:

1. Role
2. Goal
3. Context
4. Constraints
5. Format
6. Style

Give each criterion a score from 0 to 10.

Explain your reasoning.
"""
            },

            {
                "role": "user",
                "content": prompt
            }

        ]
    )

    return response.choices[0].message.content

In [ ]:
def evaluate_prompt(prompt, scenario, level):

    criteria_text = "\n".join(
        [
            f"{name}: {description}"
            for name, description in CRITERIA.items()
        ]
    )

    system_prompt = f"""
You are an expert Prompt Engineering evaluator.

Evaluate the user's prompt using these six criteria:

{criteria_text}

Scenario:
{scenario}

Difficulty:
{level}

Scoring scale:

0-2  = Missing
3-4  = Weak
5-6  = Basic
7-8  = Good
9    = Excellent
10   = Outstanding

Be objective and constructive.

For every criterion provide:
- score
- explanation

Also provide:
- strengths
- weaknesses
- missing components
- recommendations
- optimized prompt

Do NOT calculate the overall score.
Python will calculate the overall score.
"""

    response = client.chat.completions.create(

        model="openai/gpt-oss-20b",

        messages=[

            {
                "role": "system",
                "content": system_prompt
            },

            {
                "role": "user",
                "content":
                    f"Evaluate this prompt:\n\n{prompt}"
            }

        ],

        response_format={
            "type": "json_schema",

            "json_schema": {

                "name": "prompt_evaluation",

                "strict": True,

                "schema": {

                    "type": "object",

                    "properties": {

                        "scores": {

                            "type": "object",

                            "properties": {

                                "Role": {
                                    "type": "number"
                                },

                                "Goal": {
                                    "type": "number"
                                },

                                "Context": {
                                    "type": "number"
                                },

                                "Constraints": {
                                    "type": "number"
                                },

                                "Format": {
                                    "type": "number"
                                },

                                "Style": {
                                    "type": "number"
                                }

                            },

                            "required": [
                                "Role",
                                "Goal",
                                "Context",
                                "Constraints",
                                "Format",
                                "Style"
                            ],

                            "additionalProperties": False
                        },

                        "explanations": {

                            "type": "object",

                            "properties": {

                                "Role": {
                                    "type": "string"
                                },

                                "Goal": {
                                    "type": "string"
                                },

                                "Context": {
                                    "type": "string"
                                },

                                "Constraints": {
                                    "type": "string"
                                },

                                "Format": {
                                    "type": "string"
                                },

                                "Style": {
                                    "type": "string"
                                }

                            },

                            "required": [
                                "Role",
                                "Goal",
                                "Context",
                                "Constraints",
                                "Format",
                                "Style"
                            ],

                            "additionalProperties": False
                        },

                        "strengths": {

                            "type": "array",

                            "items": {
                                "type": "string"
                            }
                        },

                        "weaknesses": {

                            "type": "array",

                            "items": {
                                "type": "string"
                            }
                        },

                        "missing_components": {

                            "type": "array",

                            "items": {
                                "type": "string"
                            }
                        },

                        "recommendations": {

                            "type": "array",

                            "items": {
                                "type": "string"
                            }
                        },

                        "optimized_prompt": {

                            "type": "string"
                        }

                    },

                    "required": [
                        "scores",
                        "explanations",
                        "strengths",
                        "weaknesses",
                        "missing_components",
                        "recommendations",
                        "optimized_prompt"
                    ],

                    "additionalProperties": False
                }
            }
        },

        temperature=0.2
    )

    result = json.loads(
        response.choices[0].message.content
    )

    # Calculate overall score ourselves.
    total = sum(
        result["scores"].values()
    )

    result["overall_score"] = round(
        (total / 60) * 100
    )

    return result

In [ ]:
test_prompt = """
You are an HR manager.

Create five interview questions
for a Python developer.

Return the questions in a numbered list.
Use a professional tone.
"""

result = evaluate_prompt(
    test_prompt,
    "Generate interview questions for a Python Developer",
    "Intermediate"
)

print(
    json.dumps(
        result,
        indent=2
    )
)

In [ ]:
scores = result["scores"]

score_df = pd.DataFrame(
    {
        "Criterion": list(scores.keys()),
        "Score": list(scores.values())
    }
)

score_df

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatterpolar(
        r=list(scores.values()),
        theta=list(scores.keys()),
        fill="toself",
        name="Prompt Quality"
    )
)

fig.update_layout(
    title="Prompt Quality Radar",
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 10]
        )
    ),
    showlegend=False
)

fig.show()

In [ ]:
print("💪 STRENGTHS")
print("=" * 50)

for item in result["strengths"]:
    print("✓", item)

In [ ]:
print("⚠️ WEAKNESSES")
print("=" * 50)

for item in result["weaknesses"]:
    print("•", item)

In [ ]:
print("💡 RECOMMENDATIONS")
print("=" * 50)

for item in result["recommendations"]:
    print("→", item)

In [ ]:
print("✨ OPTIMIZED PROMPT")
print("=" * 70)

print(result["optimized_prompt"])

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import json
import os

from groq import Groq


# ==================================================
# PAGE CONFIG
# ==================================================

st.set_page_config(
    page_title="Prompt Evaluation Studio",
    page_icon="🎯",
    layout="wide"
)


# ==================================================
# API
# ==================================================

api_key = os.environ.get("GROQ_API_KEY")

if not api_key:

    st.error(
        "GROQ_API_KEY is not configured."
    )

    st.stop()


client = Groq(
    api_key=api_key
)


# ==================================================
# SCENARIOS
# ==================================================

SCENARIOS = {

    "Business": {

        "Beginner":
        "Create a professional email explaining a project delay.",

        "Intermediate":
        "Convert meeting notes into decisions, action items, owners and deadlines.",

        "Advanced":
        "Analyze quarterly business performance and produce an executive report."
    },

    "HR": {

        "Beginner":
        "Create a job description for a Python Developer.",

        "Intermediate":
        "Generate interview questions for an AI/ML Engineer.",

        "Advanced":
        "Evaluate candidate interview responses against predefined competencies."
    },

    "Marketing": {

        "Beginner":
        "Create a LinkedIn post announcing a new product.",

        "Intermediate":
        "Design a marketing campaign for a technology product.",

        "Advanced":
        "Create a complete go-to-market strategy."
    },

    "Data Analytics": {

        "Beginner":
        "Summarize monthly sales data.",

        "Intermediate":
        "Analyze sales data and identify trends and anomalies.",

        "Advanced":
        "Analyze business data and produce actionable executive insights."
    },

    "Software Development": {

        "Beginner":
        "Generate a Python factorial function.",

        "Intermediate":
        "Create a REST API using Python and FastAPI.",

        "Advanced":
        "Perform a detailed code review covering security, performance and maintainability."
    }
}


# ==================================================
# CRITERIA
# ==================================================

CRITERIA = {

    "Role":
    "Does the prompt clearly define who the AI should act as?",

    "Goal":
    "Does the prompt clearly state what the AI needs to accomplish?",

    "Context":
    "Does the prompt provide sufficient background information?",

    "Constraints":
    "Does the prompt define rules or limitations?",

    "Format":
    "Does the prompt specify the expected output structure?",

    "Style":
    "Does the prompt specify tone or communication style?"
}


# ==================================================
# SESSION STATE
# ==================================================

if "history" not in st.session_state:

    st.session_state.history = []


# ==================================================
# EVALUATION FUNCTION
# ==================================================

def evaluate_prompt(
    prompt,
    scenario,
    level
):

    criteria_text = "\n".join(
        [
            f"{k}: {v}"
            for k, v in CRITERIA.items()
        ]
    )


    system_prompt = f"""

You are an expert Prompt Engineering evaluator.

Evaluate the user's prompt using:

{criteria_text}

Scenario:
{scenario}

Difficulty:
{level}

Scoring:

0-2 = Missing
3-4 = Weak
5-6 = Basic
7-8 = Good
9 = Excellent
10 = Outstanding

Provide:

1. Six scores
2. Explanation for each score
3. Strengths
4. Weaknesses
5. Missing components
6. Recommendations
7. Optimized prompt

Do not calculate the overall score.
Python will calculate it.

"""


    response = client.chat.completions.create(

        model="openai/gpt-oss-20b",

        messages=[

            {
                "role":
                "system",

                "content":
                system_prompt
            },

            {
                "role":
                "user",

                "content":
                f"Evaluate this prompt:\n\n{prompt}"
            }
        ],

        response_format={

            "type":
            "json_schema",

            "json_schema": {

                "name":
                "prompt_evaluation",

                "strict":
                True,

                "schema": {

                    "type":
                    "object",

                    "properties": {

                        "scores": {

                            "type":
                            "object",

                            "properties": {

                                "Role":
                                {"type": "number"},

                                "Goal":
                                {"type": "number"},

                                "Context":
                                {"type": "number"},

                                "Constraints":
                                {"type": "number"},

                                "Format":
                                {"type": "number"},

                                "Style":
                                {"type": "number"}

                            },

                            "required": [
                                "Role",
                                "Goal",
                                "Context",
                                "Constraints",
                                "Format",
                                "Style"
                            ],

                            "additionalProperties":
                            False
                        },

                        "explanations": {

                            "type":
                            "object",

                            "properties": {

                                "Role":
                                {"type": "string"},

                                "Goal":
                                {"type": "string"},

                                "Context":
                                {"type": "string"},

                                "Constraints":
                                {"type": "string"},

                                "Format":
                                {"type": "string"},

                                "Style":
                                {"type": "string"}

                            },

                            "required": [
                                "Role",
                                "Goal",
                                "Context",
                                "Constraints",
                                "Format",
                                "Style"
                            ],

                            "additionalProperties":
                            False
                        },

                        "strengths": {

                            "type":
                            "array",

                            "items":
                            {"type": "string"}
                        },

                        "weaknesses": {

                            "type":
                            "array",

                            "items":
                            {"type": "string"}
                        },

                        "missing_components": {

                            "type":
                            "array",

                            "items":
                            {"type": "string"}
                        },

                        "recommendations": {

                            "type":
                            "array",

                            "items":
                            {"type": "string"}
                        },

                        "optimized_prompt": {

                            "type":
                            "string"
                        }
                    },

                    "required": [
                        "scores",
                        "explanations",
                        "strengths",
                        "weaknesses",
                        "missing_components",
                        "recommendations",
                        "optimized_prompt"
                    ],

                    "additionalProperties":
                    False
                }
            }
        },

        temperature=0.2
    )


    result = json.loads(
        response
        .choices[0]
        .message
        .content
    )


    total = sum(
        result["scores"].values()
    )


    result["overall_score"] = round(
        total / 60 * 100
    )


    return result


# ==================================================
# UI
# ==================================================

st.title(
    "🎯 AI-Powered Prompt Evaluation Studio"
)

st.write(
    "Practice → Evaluate → Improve → Master Prompt Engineering"
)


st.divider()


# ==================================================
# SCENARIO
# ==================================================

st.subheader(
    "1️⃣ Select Your Challenge"
)


col1, col2 = st.columns(2)


with col1:

    domain = st.selectbox(
        "Domain",
        list(SCENARIOS.keys())
    )


with col2:

    level = st.selectbox(
        "Difficulty",
        [
            "Beginner",
            "Intermediate",
            "Advanced"
        ]
    )


scenario = SCENARIOS[domain][level]


st.info(
    f"**Challenge:** {scenario}"
)


# ==================================================
# PROMPT
# ==================================================

st.subheader(
    "2️⃣ Write Your Prompt"
)


prompt = st.text_area(
    "Your prompt",
    height=220,
    placeholder=
    "Write your prompt here..."
)


# ==================================================
# EVALUATE
# ==================================================

if st.button(
    "🚀 Evaluate Prompt",
    use_container_width=True
):

    if not prompt.strip():

        st.warning(
            "Please enter a prompt."
        )

    else:

        with st.spinner(
            "🤖 AI Judge is evaluating..."
        ):

            try:

                result = evaluate_prompt(
                    prompt,
                    scenario,
                    level
                )


                st.session_state.latest = result


                st.session_state.history.append(
                    {
                        "attempt":
                        len(
                            st.session_state.history
                        ) + 1,

                        "score":
                        result[
                            "overall_score"
                        ]
                    }
                )


            except Exception as e:

                st.error(
                    f"Error: {e}"
                )


# ==================================================
# RESULTS
# ==================================================

if "latest" in st.session_state:

    result = st.session_state.latest

    st.divider()

    st.subheader(
        "3️⃣ Evaluation Dashboard"
    )


    score = result[
        "overall_score"
    ]


    # Scorecards

    col1, col2, col3 = st.columns(3)


    with col1:

        st.metric(
            "Overall Score",
            f"{score}/100"
        )


    with col2:

        if score >= 90:
            rating = "🏆 Excellent"

        elif score >= 75:
            rating = "🌟 Strong"

        elif score >= 60:
            rating = "👍 Good"

        elif score >= 40:
            rating = "⚠️ Needs Work"

        else:
            rating = "🔴 Beginner"


        st.metric(
            "Rating",
            rating
        )


    with col3:

        st.metric(
            "Attempts",
            len(
                st.session_state.history
            )
        )


    # ==================================================
    # SCORES
    # ==================================================

    st.subheader(
        "📊 Prompt Quality"
    )


    scores = result[
        "scores"
    ]


    col1, col2 = st.columns(2)


    with col1:

        for criterion, value in scores.items():

            st.write(
                f"**{criterion}: {value}/10**"
            )

            st.progress(
                int(value * 10)
            )


    with col2:

        fig = go.Figure()


        fig.add_trace(
            go.Scatterpolar(

                r=list(
                    scores.values()
                ),

                theta=list(
                    scores.keys()
                ),

                fill="toself",

                name="Prompt"
            )
        )


        fig.update_layout(

            polar=dict(

                radialaxis=dict(

                    visible=True,

                    range=[
                        0,
                        10
                    ]
                )
            ),

            showlegend=False
        )


        st.plotly_chart(
            fig,
            use_container_width=True
        )


    # ==================================================
    # EXPLANATIONS
    # ==================================================

    st.subheader(
        "🔍 Detailed Feedback"
    )


    for criterion in CRITERIA:

        with st.expander(
            f"{criterion} — "
            f"{scores[criterion]}/10"
        ):

            st.write(
                result[
                    "explanations"
                ][criterion]
            )


    # ==================================================
    # STRENGTHS / WEAKNESSES
    # ==================================================

    col1, col2 = st.columns(2)


    with col1:

        st.subheader(
            "💪 Strengths"
        )

        for item in result[
            "strengths"
        ]:

            st.success(
                item
            )


    with col2:

        st.subheader(
            "⚠️ Weaknesses"
        )

        for item in result[
            "weaknesses"
        ]:

            st.warning(
                item
            )


    # ==================================================
    # MISSING COMPONENTS
    # ==================================================

    st.subheader(
        "🔎 Missing Components"
    )


    if result[
        "missing_components"
    ]:

        for item in result[
            "missing_components"
        ]:

            st.error(
                item
            )

    else:

        st.success(
            "All six components are present!"
        )


    # ==================================================
    # RECOMMENDATIONS
    # ==================================================

    st.subheader(
        "💡 AI Recommendations"
    )


    for item in result[
        "recommendations"
    ]:

        st.write(
            f"➡️ {item}"
        )


    # ==================================================
    # OPTIMIZED PROMPT
    # ==================================================

    st.subheader(
        "✨ Optimized Prompt"
    )


    st.code(
        result[
            "optimized_prompt"
        ],
        language="text"
    )


    # ==================================================
    # PROGRESS
    # ==================================================

    if len(
        st.session_state.history
    ) > 1:

        st.divider()

        st.subheader(
            "📈 Prompt Improvement Journey"
        )


        history_df = pd.DataFrame(
            st.session_state.history
        )


        fig = go.Figure()


        fig.add_trace(
            go.Scatter(

                x=history_df[
                    "attempt"
                ],

                y=history_df[
                    "score"
                ],

                mode="lines+markers",

                name="Score"
            )
        )


        fig.update_layout(

            xaxis_title=
            "Attempt",

            yaxis_title=
            "Score",

            yaxis=dict(
                range=[
                    0,
                    100
                ]
            )
        )


        st.plotly_chart(
            fig,
            use_container_width=True
        )


st.divider()


st.caption(
    "Built with Python • Groq • Streamlit • Plotly • LLM-as-a-Judge"
)

In [ ]:
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print(
    "Groq API available:",
    bool(os.environ.get("GROQ_API_KEY"))
)

In [ ]:
!pkill -f streamlit || true

In [ ]:
import os
import subprocess
import sys
import time
import urllib.error
import urllib.request

log_handle = open("/content/streamlit.log", "w")
streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--browser.gatherUsageStats", "false",
    ],
    cwd="/content",
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

# Wait until the app responds before asking Colab to create its proxy URL.
for attempt in range(30):
    if streamlit_process.poll() is not None:
        log_handle.flush()
        raise RuntimeError("Streamlit exited during startup. Check /content/streamlit.log.")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8501", timeout=2) as response:
            if response.status == 200:
                print("✅ Streamlit is ready on port 8501")
                break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(1)
else:
    log_handle.flush()
    raise TimeoutError("Streamlit did not become ready. Check /content/streamlit.log.")

log_handle.flush()


In [ ]:
!cat /content/streamlit.log

In [ ]:
from getpass import getpass
from pyngrok import ngrok

# Enter your ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = getpass("Enter your ngrok authtoken: ")

if not NGROK_AUTH_TOKEN.strip():
    raise ValueError("An ngrok authtoken is required to create a public URL.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())

# Remove any old tunnels before opening a new one.
ngrok.kill()
public_url = ngrok.connect(8501, "http").public_url

print("🚀 Open your Prompt Evaluation Studio:")
print(public_url)
print("Keep this Colab runtime running while you use the URL.")
print("The URL may change when you restart the runtime or ngrok tunnel.")
